# Liveness 04 · Active Prompts (blink / turn) with MediaPipe

Active liveness defeats static photos and most replays: the app asks for **2 random prompts** from
{blink, turn left, turn right, smile} and checks them with **MediaPipe Face Landmarker** (478 landmarks +
52 blendshapes + head pose). No training needed – this notebook **tunes the thresholds** on recorded clips.

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Face_Liveness
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
import urllib.request
import mediapipe as mp, numpy as np, cv2
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

TASK = DATA / "Liveness" / "face_landmarker.task"
if not TASK.exists():
    urllib.request.urlretrieve("https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task", TASK)

landmarker = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=str(TASK)),
    output_face_blendshapes=True, output_facial_transformation_matrixes=True, num_faces=1,
    running_mode=vision.RunningMode.VIDEO))

In [ ]:
THRESHOLDS = {"blink": 0.55, "smile": 0.6, "yaw_deg": 18}

def frame_signals(result):
    if not result.face_blendshapes:
        return None
    b = {c.category_name: c.score for c in result.face_blendshapes[0]}
    m = np.array(result.facial_transformation_matrixes[0])
    yaw = float(np.degrees(np.arctan2(m[0, 2], m[2, 2])))
    return {"blink": (b["eyeBlinkLeft"] + b["eyeBlinkRight"]) / 2, "smile": (b["mouthSmileLeft"] + b["mouthSmileRight"]) / 2, "yaw": yaw}

def challenge_passed(signals, prompt):
    if prompt == "blink":
        s = [x["blink"] for x in signals]; return max(s) > THRESHOLDS["blink"] and min(s) < 0.2   # closed THEN open
    if prompt == "smile":
        return max(x["smile"] for x in signals) > THRESHOLDS["smile"]
    if prompt == "turn_left":
        return min(x["yaw"] for x in signals) < -THRESHOLDS["yaw_deg"]
    if prompt == "turn_right":
        return max(x["yaw"] for x in signals) > THRESHOLDS["yaw_deg"]
    raise ValueError(prompt)

In [ ]:
def run_clip(video_path, prompt):
    cap, signals, t = cv2.VideoCapture(str(video_path)), [], 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        s = frame_signals(landmarker.detect_for_video(image, t)); t += 33
        if s: signals.append(s)
    return bool(signals) and challenge_passed(signals, prompt)
# Tune THRESHOLDS so genuine clips pass >= 98 % and replayed / printed clips fail.